# Decoy Strategy Comparison: entity13 & living17

Сравниваем 4 стратегии формирования decoy:
- **score_coord** — случайный decoy из error pool (baseline)
- **score_coord_noise** — то же + гауссов шум
- **nearest_neighbor** — ближайший вектор из error pool в подпространстве без k-й координаты
- **nearest_neighbor_noise** — NN + гауссов шум

Тестируем на 3 тестовых наборах: `val_source`, `val_target`, одна corruption.

Decoy используется напрямую (без flow) — сравниваем качество сырых decoy.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as tv_models
import torchvision.transforms as transforms
from tqdm import tqdm
from IPython.display import display

# убедимся, что импортируем из корня проекта
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from robustness.tools.helpers import get_label_mapping
from robustness.tools import folder
from robustness.tools.breeds_helpers import make_living17, make_entity13

# Импортируем только то, что гарантированно есть в старой версии
from data_processing.negative_scores_pool import build_error_conditioned_pools

# Остальные функции инлайним — на сервере может быть старая версия без них
MIN_POOL = 50

def build_error_vector_pool(train_scores, train_labels, num_classes, min_pool=MIN_POOL):
    """Pool of full logit vectors where argmax==k AND label!=k (for NN strategy)."""
    pred_classes = train_scores.argmax(axis=1)
    pool = {}
    for k in range(num_classes):
        mask = (pred_classes == k) & (train_labels != k)
        pool[k] = train_scores[mask] if mask.sum() >= min_pool else train_scores[train_labels != k]
    return pool

def _build_decoy_score_coord(sc_np, pool_score, rng, noise_std=0.0):
    """Replace only coordinate c_hat with a random draw from error pool."""
    pred_classes = sc_np.argmax(axis=1)
    dc_np = sc_np.copy()
    for c in range(sc_np.shape[1]):
        mask = pred_classes == c
        if not mask.any():
            continue
        pool = pool_score.get(c, [])
        if len(pool) == 0:
            continue
        dc_np[mask, c] = rng.choice(pool, size=mask.sum(), replace=True)
    if noise_std > 0.0:
        dc_np = dc_np + rng.normal(0.0, noise_std, size=dc_np.shape)
    return dc_np

def _build_decoy_nearest_neighbor(sc_np, pool_error_vectors, rng, noise_std=0.0):
    """Find nearest pool vector in subspace excluding coord k; replace coord k."""
    n_samples, n_classes = sc_np.shape
    pred_classes = sc_np.argmax(axis=1)
    dc_np = sc_np.copy()
    all_coords = np.arange(n_classes)
    for k in range(n_classes):
        mask = pred_classes == k
        if not mask.any():
            continue
        pool_k = pool_error_vectors.get(k)
        if pool_k is None or len(pool_k) == 0:
            continue
        samples_k      = sc_np[mask]
        compare_coords = all_coords[all_coords != k]
        S_proj = samples_k[:, compare_coords]
        V_proj = pool_k[:,   compare_coords]
        S_sq   = (S_proj ** 2).sum(axis=1, keepdims=True)
        V_sq   = (V_proj ** 2).sum(axis=1, keepdims=True)
        dists  = np.maximum(S_sq + V_sq.T - 2.0 * (S_proj @ V_proj.T), 0.0)
        nn_idx = dists.argmin(axis=1)
        dc_np[mask, k] = pool_k[nn_idx, k]
    if noise_std > 0.0:
        dc_np = dc_np + rng.normal(0.0, noise_std, size=dc_np.shape)
    return dc_np

plt.rcParams.update({
    'font.size': 12, 'axes.titlesize': 13, 'axes.labelsize': 12,
    'xtick.labelsize': 10, 'ytick.labelsize': 10, 'legend.fontsize': 9,
})
print('Imports OK')

In [ ]:
# ─── CONFIG ────────────────────────────────────────────────────────────────────
DATA_DIR    = '/home/arina/imagenet'
DEVICE      = 'cuda:0' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE  = 256
FEATURE_DIM = 640
NOISE_STD   = 0.5

DATASETS = ['entity13', 'living17']

# 3 corruption-тестовых набора (val не грузим — долго)
TEST_CORRUPTIONS = [
    ('gaussian_noise', 3),
    ('brightness',     3),
    ('fog',            3),
]

STRATEGIES = [
    ('score_coord',            0.0),
    ('score_coord_noise',      NOISE_STD),
    ('nearest_neighbor',       0.0),
    ('nearest_neighbor_noise', NOISE_STD),
]
STRATEGY_LABELS = {
    'score_coord':            'Random (SC)',
    'score_coord_noise':      'Random + noise',
    'nearest_neighbor':       'Nearest-neighbor',
    'nearest_neighbor_noise': 'NN + noise',
}
STRATEGY_COLORS = {
    'score_coord':            '#1976D2',
    'score_coord_noise':      '#42A5F5',
    'nearest_neighbor':       '#E65100',
    'nearest_neighbor_noise': '#FF8A65',
}

print(f'Device: {DEVICE}')
print(f'Test corruptions: {[f"{c}_{s}" for c,s in TEST_CORRUPTIONS]}')

## Вспомогательные классы и функции

In [ ]:
class BREEDSClassifier(nn.Module):
    def __init__(self, num_classes, pretrained=True, feature_dim=640, freeze_backbone=False):
        super().__init__()
        backbone = tv_models.resnet50(
            weights=tv_models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
        self.backbone = nn.Sequential(*list(backbone.children())[:-1])
        if freeze_backbone:
            for p in self.backbone.parameters():
                p.requires_grad = False
        self.linear1 = nn.Linear(2048, feature_dim)
        self.linear2 = nn.Linear(feature_dim, num_classes)
        self.feature_dim = feature_dim
        self.num_classes = num_classes

    def get_features(self, x):
        return self.linear1(self.backbone(x).flatten(1))

    def forward(self, x):
        return self.linear2(F.relu(self.get_features(x)))


def get_breeds_loaders(breeds_name, data_dir, batch_size, corruptions, seed=42):
    """Возвращает (train_subset, val_subset, corr_loaders dict, num_classes).
    val_subset нужен только для проверки accuracy классификатора — в тест не идёт.
    """
    hierarchy_dir = f'{data_dir}/imagenet_class_hierarchy'
    maker = {'entity13': make_entity13, 'living17': make_living17}[breeds_name]
    ret = maker(hierarchy_dir, split='good')
    src_map = get_label_mapping('custom_imagenet', ret[1][0])
    num_classes = len(ret[1][0])

    tfm = transforms.Compose([
        transforms.Resize(256), transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.4717, 0.4499, 0.3837], [0.2600, 0.2516, 0.2575]),
    ])

    trainset = folder.ImageFolder(f'{data_dir}/imagenetv1/train/', transform=tfm, label_mapping=src_map)
    idx = np.arange(len(trainset))
    np.random.seed(seed)
    np.random.shuffle(idx)
    train_idx, val_idx = idx[:-10000], idx[-10000:]
    train_subset = torch.utils.data.Subset(trainset, train_idx)
    val_subset   = torch.utils.data.Subset(trainset, val_idx)

    def make_loader(ds):
        return torch.utils.data.DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=4)

    corr_loaders = {}
    for corr_name, corr_sev in corruptions:
        key  = f'{corr_name}_sev{corr_sev}'
        path = f'{data_dir}/imagenet-c/{corr_name}/{corr_sev}'
        if os.path.isdir(path):
            corr_loaders[key] = make_loader(
                folder.ImageFolder(path, transform=tfm, label_mapping=src_map))
        else:
            print(f'  Not found: {path}')

    return train_subset, val_subset, corr_loaders, num_classes


@torch.no_grad()
def collect_scores(model, loader_or_dataset, device, batch_size=256):
    """Собирает (scores [N,C], features [N,D], labels [N]) от модели."""
    if isinstance(loader_or_dataset, torch.utils.data.DataLoader):
        loader = loader_or_dataset
    else:
        loader = torch.utils.data.DataLoader(
            loader_or_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
    model.eval()
    sc_list, ft_list, lb_list = [], [], []
    for images, labels in tqdm(loader, leave=False):
        feats  = model.get_features(images.to(device))
        scores = model.linear2(F.relu(feats))
        sc_list.append(scores.cpu().numpy())
        ft_list.append(feats.cpu().numpy())
        lb_list.append(labels.numpy())
    return np.concatenate(sc_list), np.concatenate(ft_list), np.concatenate(lb_list)


def apply_strategy(scores_np, pool_score, pool_error_vectors, strategy, noise_std):
    """Применяет decoy стратегию к матрице scores [N, C]."""
    rng = np.random.default_rng(42)
    use_nn = strategy in ('nearest_neighbor', 'nearest_neighbor_noise')
    _noise = noise_std if strategy in ('score_coord_noise', 'nearest_neighbor_noise') else 0.0
    if use_nn:
        return _build_decoy_nearest_neighbor(scores_np, pool_error_vectors, rng, noise_std=_noise)
    else:
        return _build_decoy_score_coord(scores_np, pool_score, rng, noise_std=_noise)


print('Functions defined')

## FDR / Accuracy вычисление

In [ ]:
def compute_fdr_acc_curves(scores_np, decoy_np, labels_np, pi0=0.0):
    """
    Вычисляет q-values и accuracy curves по model scores и decoy scores.
    Возвращает dict с массивами для построения графиков.
    """
    n = len(labels_np)
    pred_scores  = scores_np.max(axis=1)
    pred_label   = scores_np.argmax(axis=1)
    decoy_scores = decoy_np.max(axis=1)
    correct      = (pred_label == labels_np).astype(int)

    # Сортируем по pred_score (возр.)
    sort_idx           = np.argsort(pred_scores)
    pred_scores_sorted = pred_scores[sort_idx]
    correct_sorted     = correct[sort_idx]

    # True FDR
    FD       = 1 - correct_sorted
    FD_CF    = np.cumsum(FD[::-1])[::-1]
    D_CF     = np.arange(n, 0, -1)
    FDR_true = np.clip(FD_CF / D_CF, 0, 1)
    QVAL_true = np.clip(np.minimum.accumulate(FDR_true), 0, 1)

    # TDC q-values
    TDC_score = np.maximum(pred_scores, decoy_scores)
    TDC_win   = (pred_scores > decoy_scores).astype(int)
    tdc_idx   = np.argsort(TDC_score)
    FD_CF_tdc = np.cumsum((1 - TDC_win[tdc_idx])[::-1])[::-1]
    D_CF_tdc  = np.maximum(np.arange(n, 0, -1) - FD_CF_tdc, 1)
    QVAL_TDC  = np.clip(np.minimum.accumulate(np.clip(FD_CF_tdc / D_CF_tdc, 0, 1)), 0, 1)

    # Mix-Max q-values
    sorted_decoys           = np.sort(decoy_scores)
    unique_z_vals, counts_z = np.unique(decoy_scores, return_counts=True)
    n_unique_z              = len(unique_z_vals)

    counts_w_leq_z = np.searchsorted(pred_scores_sorted, unique_z_vals, side='left')
    counts_z_leq_z = np.searchsorted(sorted_decoys,      unique_z_vals, side='left')
    P_W_leq_z = np.clip((counts_w_leq_z - pi0 * counts_z_leq_z) / ((1 - pi0) * n), 0, 1)
    P_Y_leq_z = np.clip(counts_z_leq_z / n, 0, 1)
    R_j = np.clip(
        np.divide(P_W_leq_z, P_Y_leq_z,
                  out=np.zeros_like(P_W_leq_z), where=P_Y_leq_z > 0), 0, 1)

    fdr_values = np.zeros(n)
    for i, T in enumerate(pred_scores_sorted[::-1]):
        D     = i + 1
        F_0   = pi0 * np.sum(decoy_scores > T)
        z_idx = np.searchsorted(unique_z_vals, T, side='left')
        F_1   = 0.0 if z_idx >= n_unique_z else (
            (1 - pi0) * np.sum(R_j[z_idx:] * counts_z[z_idx:]))
        fdr_values[i] = (F_0 + F_1) / D if D > 0 else 0.0

    QVAL_mixmax = np.clip(
        np.minimum.accumulate(np.clip(fdr_values, 0, 1)[::-1]), 0, 1)

    # Accuracy curves
    pi0_tdc = float(np.clip(QVAL_TDC[0],    0, 1))
    pi0_mm  = float(np.clip(QVAL_mixmax[0], 0, 1))

    Acc_true   = np.zeros(n)
    Acc_est    = np.zeros(n)   # TDC
    Acc_est_MM = np.zeros(n)   # Mix-Max
    for i in range(n):
        TP_true     = correct_sorted[i:].sum()
        TN_true     = (1 - correct_sorted[:i]).sum()
        Acc_true[i] = (TP_true + TN_true) / n

        accepted      = n - i
        FP_tdc        = accepted * QVAL_TDC[i]
        TP_tdc        = accepted * (1 - QVAL_TDC[i])
        TN_tdc        = n * pi0_tdc - FP_tdc
        Acc_est[i]    = np.clip((TP_tdc + TN_tdc) / n, 0, 1)

        FP_mm         = accepted * QVAL_mixmax[i]
        TP_mm         = accepted * (1 - QVAL_mixmax[i])
        TN_mm         = n * pi0_mm - FP_mm
        Acc_est_MM[i] = np.clip((TP_mm + TN_mm) / n, 0, 1)

    Acc_true = np.clip(Acc_true, 0, 1)
    normalized_rank = np.arange(n) / n

    total_TP  = int(correct_sorted.sum())
    TP_from_i = np.cumsum(correct_sorted[::-1])[::-1]
    D_from_i  = np.arange(n, 0, -1)

    true_acc_full = float(correct.mean())
    acc_st_true   = float(Acc_true[0])
    acc_ta_true   = float(Acc_true.max())
    acc_st_est_mm = float(Acc_est_MM[0])
    acc_ta_est_mm = float(Acc_est_MM.max())

    return dict(
        normalized_rank  = normalized_rank,
        pred_scores_sorted = pred_scores_sorted,
        pred_scores      = pred_scores,
        decoy_scores     = decoy_scores,
        QVAL_true        = QVAL_true,
        QVAL_TDC         = QVAL_TDC,
        QVAL_mixmax      = QVAL_mixmax,
        Acc_true         = Acc_true,
        Acc_est          = Acc_est,
        Acc_est_MM       = Acc_est_MM,
        precision_true   = np.where(D_from_i > 0, TP_from_i / D_from_i, 0),
        recall_true      = TP_from_i / max(total_TP, 1),
        precision_est    = np.clip(1 - QVAL_mixmax, 0, 1),
        recall_est       = np.clip((1 - QVAL_mixmax) * D_from_i / max(total_TP, 1), 0, 1),
        correct          = correct,
        pred_label       = pred_label,
        labels           = labels_np,
        true_acc         = true_acc_full,
        acc_st_true      = acc_st_true,
        acc_ta_true      = acc_ta_true,
        acc_st_est_mm    = acc_st_est_mm,
        acc_ta_est_mm    = acc_ta_est_mm,
        err_st_mm        = abs(acc_st_est_mm - acc_st_true),
        err_ta_mm        = abs(acc_ta_est_mm - acc_ta_true),
        n                = n,
    )


print('FDR computation functions defined')

## Загрузка данных и моделей

In [ ]:
all_data = {}

for breeds_name in DATASETS:
    print(f'\n{"="*60}')
    print(f'DATASET: {breeds_name}')
    print(f'{"="*60}')

    train_subset, val_subset, corr_loaders, num_classes = get_breeds_loaders(
        breeds_name, DATA_DIR, BATCH_SIZE, TEST_CORRUPTIONS)
    print(f'  num_classes={num_classes},  train={len(train_subset)}')
    print(f'  Corruption loaders: {list(corr_loaders.keys())}')

    # Загружаем / обучаем классификатор
    ckpt_path = f'breeds_{breeds_name}_classifier.pth'
    model = BREEDSClassifier(num_classes=num_classes, pretrained=True,
                              feature_dim=FEATURE_DIM).to(DEVICE)
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE, weights_only=False))
        print(f'  Classifier loaded from {ckpt_path}')
    else:
        print('  Checkpoint not found — training from scratch...')
        from make_breeds import train_breeds_classifier
        val_loader = torch.utils.data.DataLoader(
            val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
        train_loader = torch.utils.data.DataLoader(
            train_subset, batch_size=64, shuffle=True, num_workers=4)
        train_breeds_classifier(
            model, train_loader, val_loader,
            epochs=30, lr=0.01, device=DEVICE,
            save_path=ckpt_path, patience=5)
    model.eval()

    # Собираем train логиты + features для пулов и flow
    print('  Collecting train scores...')
    train_scores, train_feats, train_labels = collect_scores(model, train_subset, DEVICE)
    print(f'  Train: {train_scores.shape}  '
          f'acc={(train_scores.argmax(1) == train_labels).mean():.4f}')

    # Строим пулы (по всему train)
    pool_score, pool_vectors = build_error_conditioned_pools(
        train_scores, train_labels, num_classes, verbose=False)
    pool_error_vectors = build_error_vector_pool(
        train_scores, train_labels, num_classes)

    # Собираем логиты для corruption тестовых наборов
    test_sets_raw = {}
    for tset_name, loader in corr_loaders.items():
        print(f'  {tset_name} ...')
        sc, ft, lb = collect_scores(model, loader, DEVICE)
        test_sets_raw[tset_name] = (sc, ft, lb)
        print(f'    n={len(lb)},  acc={(sc.argmax(1) == lb).mean():.4f}')

    all_data[breeds_name] = dict(
        model              = model,
        num_classes        = num_classes,
        pool_score         = pool_score,
        pool_error_vectors = pool_error_vectors,
        train_scores       = train_scores,
        train_feats        = train_feats,
        train_labels       = train_labels,
        test_sets_raw      = test_sets_raw,
    )
    print(f'  Done: {list(test_sets_raw.keys())}')

## Применяем стратегии и вычисляем метрики

In [ ]:
# Для каждого датасета × тестового набора × стратегии — строим decoy и считаем FDR/Acc

results = {}  # [breeds_name][testset_name][strategy_name] -> curve dict

for breeds_name, bdata in all_data.items():
    results[breeds_name] = {}
    pool_score         = bdata['pool_score']
    pool_error_vectors = bdata['pool_error_vectors']

    for tset_name, (sc_np, ft_np, lb_np) in bdata['test_sets_raw'].items():
        results[breeds_name][tset_name] = {}
        for strat_name, noise_std in STRATEGIES:
            decoy_np = apply_strategy(
                sc_np, pool_score, pool_error_vectors, strat_name, noise_std)
            curves = compute_fdr_acc_curves(sc_np, decoy_np, lb_np)
            results[breeds_name][tset_name][strat_name] = curves
            print(f'  {breeds_name}/{tset_name}/{strat_name}:  '
                  f'true_acc={curves["true_acc"]:.3f}  '
                  f'err_st={curves["err_st_mm"]:.3f}  '
                  f'err_ta={curves["err_ta_mm"]:.3f}')

print('\nAll decoys computed.')

## Тренировка flow для каждой стратегии

Для каждой из 4 стратегий обучаем отдельный `ScoreShiftFlowWrapper` (архитектура seed_42: n_flows=12, encoder_dim=128).  
Checkpoint сохраняется в `breeds_{name}_flow_{strategy}.pth` — при повторном запуске загружается без переобучения.  
После обучения `generate_decoys` генерирует decoy через flow для всех тестовых наборов.

In [ ]:
# ── MAE сводная таблица: Raw vs Flow ───────────────────────────────────────────
rows_flow = []
for breeds_name, strat_dict in flow_results.items():
    for strat_name, tset_dict in strat_dict.items():
        for tset_name, c in tset_dict.items():
            rows_flow.append(dict(
                dataset   = breeds_name,
                testset   = tset_name,
                strategy  = STRATEGY_LABELS[strat_name],
                mode      = 'flow',
                err_st    = c['err_st_mm'],
                err_ta    = c['err_ta_mm'],
                true_acc  = c['true_acc'],
            ))

rows_raw = []
for breeds_name, tset_dict in results.items():
    for tset_name, strat_dict in tset_dict.items():
        for strat_name, c in strat_dict.items():
            rows_raw.append(dict(
                dataset  = breeds_name,
                testset  = tset_name,
                strategy = STRATEGY_LABELS[strat_name],
                mode     = 'raw',
                err_st   = c['err_st_mm'],
                err_ta   = c['err_ta_mm'],
                true_acc = c['true_acc'],
            ))

compare_df = pd.concat([pd.DataFrame(rows_raw), pd.DataFrame(rows_flow)], ignore_index=True)

# Таблица: среднее MAE_ST и MAE_TA по strategy × mode
pivot_compare = compare_df.groupby(['strategy', 'mode'])[['err_st', 'err_ta']].mean().round(4)
print('Mean MAE — Raw vs Flow по стратегиям:')
display(pivot_compare)

# Bar chart: Raw vs Flow MAE_ST
strat_order = [STRATEGY_LABELS[s] for s, _ in STRATEGIES]
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, metric, title in [
    (axes[0], 'err_st', 'MAE_ST'),
    (axes[1], 'err_ta', 'MAE_TA'),
]:
    x = np.arange(len(strat_order))
    w = 0.35
    raw_means  = [compare_df[(compare_df['strategy']==s) & (compare_df['mode']=='raw')][metric].mean()
                  for s in strat_order]
    flow_means = [compare_df[(compare_df['strategy']==s) & (compare_df['mode']=='flow')][metric].mean()
                  for s in strat_order]
    raw_stds   = [compare_df[(compare_df['strategy']==s) & (compare_df['mode']=='raw')][metric].std()
                  for s in strat_order]
    flow_stds  = [compare_df[(compare_df['strategy']==s) & (compare_df['mode']=='flow')][metric].std()
                  for s in strat_order]

    bars_r = ax.bar(x - w/2, raw_means,  w, yerr=raw_stds,  label='Raw pool',
                    color='#90CAF9', capsize=4, alpha=0.9, edgecolor='black', linewidth=0.7)
    bars_f = ax.bar(x + w/2, flow_means, w, yerr=flow_stds, label='Flow',
                    color='#1976D2', capsize=4, alpha=0.9, edgecolor='black', linewidth=0.7)

    for bar, val in zip(list(bars_r) + list(bars_f), raw_means + flow_means):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(strat_order, rotation=15, ha='right')
    ax.set_ylabel('Mean Absolute Error')
    ax.set_title(f'{title} — Raw pool vs Flow')
    ax.legend(fontsize=9)
    ax.grid(axis='y', ls='--', alpha=0.4)
    ax.set_ylim(bottom=0)

fig.suptitle('MAE comparison: Raw pool decoy vs Flow-generated decoy', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Raw vs Flow: прямое сравнение score distributions ──────────────────────────
# Для каждого датасета × тестового набора: 2 строки (raw / flow), 4 колонки (стратегии)

for breeds_name in DATASETS:
    for tset_name in all_data[breeds_name]['test_sets_raw']:
        sc_np, ft_np, lb_np = all_data[breeds_name]['test_sets_raw'][tset_name]
        strat_names = [s for s, _ in STRATEGIES]
        n_strats    = len(strat_names)

        fig, axes = plt.subplots(2, n_strats, figsize=(5 * n_strats, 8), sharey=False)
        all_scores_ref = sc_np.max(axis=1)
        bins = np.linspace(all_scores_ref.min() - 0.3, all_scores_ref.max() + 0.3, 60)

        for col, strat_name in enumerate(strat_names):
            color = STRATEGY_COLORS[strat_name]
            label = STRATEGY_LABELS[strat_name]

            raw_c  = results[breeds_name][tset_name][strat_name]
            flow_c = flow_results_by_tset.get(breeds_name, {}).get(tset_name, {}).get(strat_name)

            # Row 0: raw decoy
            ax = axes[0][col]
            sns.histplot(raw_c['pred_scores'],  bins=bins, stat='density',
                         color='steelblue', kde=True, fill=True, alpha=0.3, label='model', ax=ax)
            sns.histplot(raw_c['decoy_scores'], bins=bins, stat='density',
                         color=color, kde=True, fill=True, alpha=0.4, label='decoy (raw)', ax=ax)
            ax.set_title(f'{label}\nRAW  err_st={raw_c["err_st_mm"]:.3f}', fontsize=10)
            ax.set_xlabel('Max logit')
            if col == 0: ax.set_ylabel('Density (raw)')
            ax.legend(fontsize=7)

            # Row 1: flow decoy
            ax = axes[1][col]
            if flow_c is not None:
                sns.histplot(flow_c['pred_scores'],  bins=bins, stat='density',
                             color='steelblue', kde=True, fill=True, alpha=0.3, label='model', ax=ax)
                sns.histplot(flow_c['decoy_scores'], bins=bins, stat='density',
                             color=color, kde=True, fill=True, alpha=0.4, label='decoy (flow)', ax=ax)
                ax.set_title(f'FLOW  err_st={flow_c["err_st_mm"]:.3f}', fontsize=10)
                ax.legend(fontsize=7)
            else:
                ax.set_title('FLOW (no data)')
            ax.set_xlabel('Max logit')
            if col == 0: ax.set_ylabel('Density (flow)')

        fig.suptitle(f'{breeds_name} | {tset_name}  —  Raw vs Flow score distributions', fontsize=13)
        plt.tight_layout()
        plt.show()

In [ ]:
# ── Flow: сводный dashboard по стратегиям (FDR + Acc, аналог raw dashboard) ──
# flow_results индексируется [breeds][strategy][testset], перекладываем в [breeds][testset][strategy]
flow_results_by_tset = {}
for breeds_name, strat_dict in flow_results.items():
    flow_results_by_tset[breeds_name] = {}
    for strat_name, tset_dict in strat_dict.items():
        for tset_name, curves in tset_dict.items():
            flow_results_by_tset[breeds_name].setdefault(tset_name, {})[strat_name] = curves

# Dashboard: FDR + Acc через flow (аналог Визуализации 9 для raw)
for breeds_name in DATASETS:
    for tset_name in all_data[breeds_name]['test_sets_raw']:
        if tset_name not in flow_results_by_tset.get(breeds_name, {}):
            continue
        tset_results = flow_results_by_tset[breeds_name][tset_name]
        strat_names  = list(tset_results.keys())
        n_strats     = len(strat_names)

        fig, axes = plt.subplots(2, n_strats, figsize=(5 * n_strats, 8))
        if n_strats == 1:
            axes = axes.reshape(2, 1)

        ref = list(tset_results.values())[0]
        r   = ref['normalized_rank']

        for col, strat_name in enumerate(strat_names):
            c     = tset_results[strat_name]
            color = STRATEGY_COLORS[strat_name]

            # Row 0: FDR
            ax = axes[0][col]
            ax.plot(r, c['QVAL_true'],   color='gray',  lw=1.5, ls='--', label='True FDR')
            ax.plot(r, c['QVAL_mixmax'], color=color,   lw=1.8,          label='MixMax (flow)')
            ax.plot(r, c['QVAL_TDC'],    color='navy',  lw=1.2, ls=':',  label='TDC (flow)')
            ax.axhline(0.1, color='black', lw=0.7, ls=':', alpha=0.4)
            ax.set_title(f'{STRATEGY_LABELS[strat_name]}\nMAE_ST={c["err_st_mm"]:.3f}', fontsize=11)
            ax.set_xlabel('Fraction accepted')
            ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)
            if col == 0: ax.set_ylabel('q-value (FDR)')

            # Row 1: Accuracy
            ax = axes[1][col]
            ax.plot(r, c['Acc_true'],   color='black', lw=2,   ls='--', label='True Acc')
            ax.plot(r, c['Acc_est_MM'], color=color,   lw=1.8,          label='ENGPE-TA (flow)')
            ax.plot(r, c['Acc_est'],    color='navy',  lw=1.2, ls=':',  label='ENGPE (flow)')
            ax.set_title(f'Accuracy  MAE_TA={c["err_ta_mm"]:.3f}', fontsize=10)
            ax.set_xlabel('Fraction accepted')
            ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)
            if col == 0: ax.set_ylabel('Accuracy')

        fig.suptitle(f'{breeds_name} | {tset_name}  —  Flow-generated decoys', fontsize=13)
        plt.tight_layout()
        plt.show()

### Результаты flow: FDR + Accuracy по стратегиям

`flow_results[breeds_name][strategy_name][testset_name]` — те же поля, что и `results`, но decoy сгенерированы flow, а не напрямую из пула.

In [ ]:
from data_processing.score_feature_dataset import ScoreFeatureDataset
from flows.flow_FN import ScoreShiftFlowWrapper

# flow_results[breeds_name][strategy_name][testset_name] -> curve dict (flow-generated decoys)
flow_results = {}

FLOW_EPOCHS    = 30
FLOW_LR        = 3e-4
FLOW_PATIENCE  = 5
FLOW_N         = 12    # coupling layers
FLOW_ENC_DIM   = 128
FLOW_SUBSAMPLE = 0.5   # доля train данных для обучения flow (ускорение для проверки гипотез)
FLOW_SEED      = 42

for breeds_name, bdata in all_data.items():
    flow_results[breeds_name] = {}
    num_classes        = bdata['num_classes']
    pool_score         = bdata['pool_score']
    pool_error_vectors = bdata['pool_error_vectors']
    train_scores       = bdata['train_scores']
    train_feats        = bdata['train_feats']
    train_labels       = bdata['train_labels']

    # ── Субсемплинг 50% для обучения flow ────────────────────────────────────
    n_total  = len(train_scores)
    n_flow   = int(n_total * FLOW_SUBSAMPLE)
    rng_sub  = np.random.default_rng(FLOW_SEED)
    sub_idx  = rng_sub.choice(n_total, size=n_flow, replace=False)
    sub_idx.sort()

    flow_train_scores = train_scores[sub_idx]
    flow_train_feats  = train_feats[sub_idx]
    flow_train_labels = train_labels[sub_idx]
    print(f'\n{breeds_name}: flow train subset {n_flow}/{n_total} '
          f'({FLOW_SUBSAMPLE*100:.0f}%)  '
          f'acc={(flow_train_scores.argmax(1) == flow_train_labels).mean():.4f}')

    for strat_name, noise_std in STRATEGIES:
        print(f'\n{"#"*60}')
        print(f'  {breeds_name}  |  strategy: {strat_name}  (noise={noise_std})')
        print(f'{"#"*60}')

        # ── Строим decoy для train subset ─────────────────────────────────────
        train_decoy = apply_strategy(
            flow_train_scores, pool_score, pool_error_vectors, strat_name, noise_std)

        train_ds = ScoreFeatureDataset(
            torch.from_numpy(flow_train_scores).float(),
            torch.from_numpy(flow_train_feats).float(),
            torch.from_numpy(train_decoy).float(),
            torch.from_numpy(flow_train_labels).long(),
        )

        # ── Загружаем или обучаем flow ────────────────────────────────────────
        flow_path = f'breeds_{breeds_name}_flow_{strat_name}_half.pth'
        flow = ScoreShiftFlowWrapper(
            num_classes=num_classes,
            n_flows=FLOW_N,
            feature_dim=FEATURE_DIM,
            hidden_dim=256,
            encoder_dim=FLOW_ENC_DIM,
            clip_val=5.0,
        ).to(DEVICE)

        if os.path.exists(flow_path):
            flow.load_state_dict(torch.load(flow_path, map_location=DEVICE, weights_only=False))
            print(f'  Flow loaded from {flow_path}')
        else:
            print(f'  Training flow on {n_flow} samples → {flow_path}')
            flow.train_flow(
                train_ds,
                epochs=FLOW_EPOCHS,
                lr=FLOW_LR,
                batch_size=256,
                device=DEVICE,
                patience=FLOW_PATIENCE,
                grad_clip=1.0,
            )
            torch.save(flow.state_dict(), flow_path)
            print(f'  Saved → {flow_path}')

        flow.eval()

        # ── Генерируем decoy через flow для каждого тестового набора ─────────
        flow_results[breeds_name].setdefault(strat_name, {})

        for tset_name, (sc_np, ft_np, lb_np) in bdata['test_sets_raw'].items():
            tset_decoy_pool = apply_strategy(
                sc_np, pool_score, pool_error_vectors, strat_name, noise_std)

            test_ds = ScoreFeatureDataset(
                torch.from_numpy(sc_np).float(),
                torch.from_numpy(ft_np).float(),
                torch.from_numpy(tset_decoy_pool).float(),
                torch.from_numpy(lb_np).long(),
            )

            ms_np, ds_flow_np, ls_np = flow.generate_decoys(test_ds, device=DEVICE)
            curves = compute_fdr_acc_curves(ms_np, ds_flow_np, ls_np)
            flow_results[breeds_name][strat_name][tset_name] = curves

            print(f'    {tset_name}: true_acc={curves["true_acc"]:.3f}  '
                  f'err_st={curves["err_st_mm"]:.3f}  err_ta={curves["err_ta_mm"]:.3f}')

print('\nAll flow experiments done.')

## Визуализация 1: Распределения score (model vs decoy)

In [ ]:
def plot_score_distributions(breeds_name, results_dict, testset_name):
    tset_results = results_dict[breeds_name][testset_name]
    strat_names  = list(tset_results.keys())
    n_strats     = len(strat_names)

    fig, axes = plt.subplots(1, n_strats, figsize=(5 * n_strats, 4), sharey=False)
    if n_strats == 1:
        axes = [axes]

    ref_curves = tset_results[strat_names[0]]
    all_scores = ref_curves['pred_scores']
    bins = np.linspace(all_scores.min() - 0.3, all_scores.max() + 0.3, 60)
    incorrect_mask = ref_curves['correct'] == 0

    for ax, strat_name in zip(axes, strat_names):
        c = tset_results[strat_name]
        color = STRATEGY_COLORS[strat_name]

        sns.histplot(c['pred_scores'],  bins=bins, stat='density', color='steelblue',
                     kde=True, fill=True, alpha=0.3, label='model scores', ax=ax)
        sns.histplot(c['decoy_scores'], bins=bins, stat='density', color=color,
                     kde=True, fill=True, alpha=0.4, label='decoy (null)', ax=ax)
        if incorrect_mask.any():
            sns.histplot(c['pred_scores'][incorrect_mask], bins=bins, stat='density',
                         color='crimson', kde=True, fill=True, alpha=0.3,
                         label='incorrect', ax=ax)

        ax.set_title(f'{STRATEGY_LABELS[strat_name]}\ntrue_acc={c["true_acc"]:.3f}')
        ax.set_xlabel('Max logit (score)')
        if ax is axes[0]:
            ax.set_ylabel('Density')
        ax.legend(fontsize=8)

    fig.suptitle(f'{breeds_name} | {testset_name} — Score Distributions', fontsize=13)
    plt.tight_layout()
    plt.show()


for breeds_name in DATASETS:
    for tset_name in all_data[breeds_name]['test_sets_raw']:
        plot_score_distributions(breeds_name, results, tset_name)

## Визуализация 2: ECDF — model vs decoy

In [ ]:
def plot_ecdf_comparison(breeds_name, results_dict, testset_name):
    tset_results = results_dict[breeds_name][testset_name]
    strat_names  = list(tset_results.keys())
    n_strats     = len(strat_names)

    fig, axes = plt.subplots(1, n_strats, figsize=(5 * n_strats, 4), sharey=True)
    if n_strats == 1:
        axes = [axes]

    for ax, strat_name in zip(axes, strat_names):
        c   = tset_results[strat_name]
        n   = c['n']
        cdf = np.arange(1, n + 1) / n

        ax.plot(np.sort(c['pred_scores']),  cdf, color='steelblue', lw=1.8, label='model scores')
        ax.plot(np.sort(c['decoy_scores']), cdf,
                color=STRATEGY_COLORS[strat_name], lw=1.8, ls='--', label='decoy (null)')
        ax.set_title(STRATEGY_LABELS[strat_name])
        ax.set_xlabel('Max logit')
        if ax is axes[0]:
            ax.set_ylabel('Cumulative proportion')
        ax.legend(fontsize=8)
        ax.grid(ls='--', alpha=0.4)

    fig.suptitle(f'{breeds_name} | {testset_name} — ECDF', fontsize=13)
    plt.tight_layout()
    plt.show()


for breeds_name in DATASETS:
    for tset_name in all_data[breeds_name]['test_sets_raw']:
        plot_ecdf_comparison(breeds_name, results, tset_name)

## Визуализация 3: Scatter — decoy score vs model score

In [ ]:
def plot_score_scatter(breeds_name, results_dict, testset_name, max_points=2000):
    """Scatter: decoy_score (y) vs model_score (x), раскрашено по correct/incorrect."""
    tset_results = results_dict[breeds_name][testset_name]
    strat_names  = list(tset_results.keys())
    n_strats     = len(strat_names)

    fig, axes = plt.subplots(1, n_strats, figsize=(5 * n_strats, 4), sharey=False)
    if n_strats == 1:
        axes = [axes]

    for ax, strat_name in zip(axes, strat_names):
        c       = tset_results[strat_name]
        ms      = c['pred_scores']
        ds      = c['decoy_scores']
        correct = c['correct']

        # Subsample for speed
        idx = np.random.choice(len(ms), min(max_points, len(ms)), replace=False)
        ms_s, ds_s, cor_s = ms[idx], ds[idx], correct[idx]

        ax.scatter(ms_s[cor_s == 1], ds_s[cor_s == 1],
                   s=8, alpha=0.3, color='steelblue', label='correct', rasterized=True)
        ax.scatter(ms_s[cor_s == 0], ds_s[cor_s == 0],
                   s=8, alpha=0.5, color='crimson', label='incorrect', rasterized=True)

        lims = [min(ms.min(), ds.min()) - 0.1, max(ms.max(), ds.max()) + 0.1]
        ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5)
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel('Model score (max logit)')
        if ax is axes[0]:
            ax.set_ylabel('Decoy score (max logit)')
        ax.set_title(STRATEGY_LABELS[strat_name])
        ax.legend(fontsize=8)
        ax.grid(ls='--', alpha=0.3)

    fig.suptitle(f'{breeds_name} | {testset_name} — Decoy vs Model Score', fontsize=13)
    plt.tight_layout()
    plt.show()


for breeds_name in DATASETS:
    for tset_name in all_data[breeds_name]['test_sets_raw']:
        plot_score_scatter(breeds_name, results, tset_name)

## Визуализация 4: FDR кривые — True vs MixMax vs TDC

In [ ]:
def plot_fdr_curves(breeds_name, results_dict, testset_name):
    tset_results = results_dict[breeds_name][testset_name]
    strat_names  = list(tset_results.keys())
    n_strats     = len(strat_names)

    fig, axes = plt.subplots(1, n_strats, figsize=(5 * n_strats, 4), sharey=True)
    if n_strats == 1:
        axes = [axes]

    for ax, strat_name in zip(axes, strat_names):
        c = tset_results[strat_name]
        r = c['normalized_rank']

        ax.plot(r, c['QVAL_true'],    color='gray',    lw=1.5, ls='--', label='True FDR')
        ax.plot(r, c['QVAL_mixmax'],  color='#E65100', lw=1.8,          label='Mix-Max FDR')
        ax.plot(r, c['QVAL_TDC'],     color='#1976D2', lw=1.4, ls=':',  label='TDC FDR')
        ax.axhline(0.1, color='black', lw=0.8, ls=':',  alpha=0.4, label='FDR=0.1')
        ax.axhline(0.2, color='black', lw=0.8, ls='--', alpha=0.3)

        err_st = c['err_st_mm']
        ax.set_title(f'{STRATEGY_LABELS[strat_name]}\nMAE_ST={err_st:.3f}')
        ax.set_xlabel('Fraction accepted')
        if ax is axes[0]:
            ax.set_ylabel('q-value (FDR)')
        ax.legend(fontsize=8)
        ax.grid(ls='--', alpha=0.4)
        ax.set_ylim(0, 1.05)

    fig.suptitle(f'{breeds_name} | {testset_name} — FDR curves', fontsize=13)
    plt.tight_layout()
    plt.show()


for breeds_name in DATASETS:
    for tset_name in all_data[breeds_name]['test_sets_raw']:
        plot_fdr_curves(breeds_name, results, tset_name)

## Визуализация 5: Accuracy curves — все стратегии на одном графике

In [ ]:
def plot_accuracy_curves_overlay(breeds_name, results_dict, testset_name):
    """Все стратегии на одном графике (acc_est_MM vs acc_true)."""
    tset_results = results_dict[breeds_name][testset_name]

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Left: acc curves overlay
    ax = axes[0]
    ref = list(tset_results.values())[0]
    r   = ref['normalized_rank']
    ax.plot(r, ref['Acc_true'], color='black', lw=2, ls='--', label='True Acc', zorder=5)

    for strat_name, c in tset_results.items():
        color = STRATEGY_COLORS[strat_name]
        ax.plot(r, c['Acc_est_MM'], color=color, lw=1.8,
                label=f'{STRATEGY_LABELS[strat_name]} (MAE={c["err_st_mm"]:.3f})')

    ax.set_xlabel('Fraction accepted')
    ax.set_ylabel('Accuracy')
    ax.set_title('ENGPE-TA Accuracy curves')
    ax.legend(fontsize=8)
    ax.grid(ls='--', alpha=0.4)

    # Right: FDR overlay
    ax = axes[1]
    ax.plot(r, ref['QVAL_true'], color='black', lw=2, ls='--', label='True FDR', zorder=5)
    for strat_name, c in tset_results.items():
        color = STRATEGY_COLORS[strat_name]
        ax.plot(r, c['QVAL_mixmax'], color=color, lw=1.8,
                label=STRATEGY_LABELS[strat_name])

    ax.set_xlabel('Fraction accepted')
    ax.set_ylabel('q-value (FDR)')
    ax.set_title('Mix-Max FDR curves')
    ax.legend(fontsize=8)
    ax.grid(ls='--', alpha=0.4)
    ax.set_ylim(0, 1.05)

    fig.suptitle(f'{breeds_name} | {testset_name} — Strategy comparison', fontsize=13)
    plt.tight_layout()
    plt.show()


for breeds_name in DATASETS:
    for tset_name in all_data[breeds_name]['test_sets_raw']:
        plot_accuracy_curves_overlay(breeds_name, results, tset_name)

## Визуализация 6: Precision-Recall кривые

In [ ]:
def plot_precision_recall(breeds_name, results_dict, testset_name):
    tset_results = results_dict[breeds_name][testset_name]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # True P-R
    ax = axes[0]
    ref = list(tset_results.values())[0]
    ax.plot(ref['recall_true'], ref['precision_true'],
            color='black', lw=2, ls='--', label='True')
    for strat_name, c in tset_results.items():
        ax.plot(c['recall_true'], c['precision_true'],
                color=STRATEGY_COLORS[strat_name], lw=1.2, alpha=0.6,
                label=STRATEGY_LABELS[strat_name])
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall (true)')
    ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.4)

    # Estimated P-R
    ax = axes[1]
    ax.plot(ref['recall_true'], ref['precision_true'],
            color='black', lw=2, ls='--', label='True')
    for strat_name, c in tset_results.items():
        ax.plot(c['recall_est'], c['precision_est'],
                color=STRATEGY_COLORS[strat_name], lw=1.8,
                label=STRATEGY_LABELS[strat_name])
    ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
    ax.set_title('Precision-Recall (MixMax estimated)')
    ax.legend(fontsize=8); ax.grid(ls='--', alpha=0.4)

    fig.suptitle(f'{breeds_name} | {testset_name} — Precision-Recall', fontsize=13)
    plt.tight_layout()
    plt.show()


for breeds_name in DATASETS:
    for tset_name in all_data[breeds_name]['test_sets_raw']:
        plot_precision_recall(breeds_name, results, tset_name)

## Визуализация 7: Scatter — estimated vs true accuracy (по тестовым наборам)

In [ ]:
def plot_est_vs_true_accuracy(results_dict):
    """Scatter: est_accuracy vs true_accuracy для каждой стратегии.
    Точка = один тестовый набор из одного датасета."""

    fig, axes = plt.subplots(1, len(STRATEGIES), figsize=(5 * len(STRATEGIES), 4), sharey=True)

    markers = {'entity13': 'o', 'living17': 's'}
    marker_colors = {'entity13': '#1976D2', 'living17': '#E65100'}

    for ax, (strat_name, _) in zip(axes, STRATEGIES):
        all_true_st, all_est_st = [], []
        all_true_ta, all_est_ta = [], []

        for breeds_name, tset_dict in results_dict.items():
            for tset_name, strat_dict in tset_dict.items():
                c = strat_dict[strat_name]
                ax.scatter(c['acc_st_true'], c['acc_st_est_mm'],
                           marker=markers.get(breeds_name, 'o'),
                           color=marker_colors.get(breeds_name, 'gray'),
                           s=80, alpha=0.8, zorder=3,
                           label=f'{breeds_name}/{tset_name}' if strat_name == STRATEGIES[0][0] else '')
                all_true_st.append(c['acc_st_true'])
                all_est_st.append(c['acc_st_est_mm'])

        all_v = all_true_st + all_est_st
        if all_v:
            lims = [min(all_v) - 0.02, max(all_v) + 0.02]
            ax.plot(lims, lims, 'k--', lw=0.8, alpha=0.5)
            ax.set_xlim(lims); ax.set_ylim(lims)

        mae = np.mean([abs(e - t) for e, t in zip(all_est_st, all_true_st)]) if all_true_st else float('nan')
        ax.set_title(f'{STRATEGY_LABELS[strat_name]}\nMAE={mae:.3f}')
        ax.set_xlabel('True accuracy (ST)')
        if ax is axes[0]:
            ax.set_ylabel('Estimated accuracy (ST)')
        ax.grid(ls='--', alpha=0.4)

    # Единая легенда
    from matplotlib.lines import Line2D
    legend_elements = [
        Line2D([0], [0], marker='o', color='w', markerfacecolor='#1976D2', markersize=9, label='entity13'),
        Line2D([0], [0], marker='s', color='w', markerfacecolor='#E65100', markersize=9, label='living17'),
    ]
    axes[-1].legend(handles=legend_elements, fontsize=9, loc='upper left')

    fig.suptitle('Est. vs True Accuracy (ST) — all datasets × test sets', fontsize=13)
    plt.tight_layout()
    plt.show()


plot_est_vs_true_accuracy(results)

## Визуализация 8: MAE сводная таблица и bar chart

In [ ]:
# Сводная таблица MAE
rows = []
for breeds_name, tset_dict in results.items():
    for tset_name, strat_dict in tset_dict.items():
        for strat_name, c in strat_dict.items():
            rows.append(dict(
                dataset=breeds_name,
                testset=tset_name,
                strategy=STRATEGY_LABELS[strat_name],
                true_acc=c['true_acc'],
                acc_st_true=c['acc_st_true'],
                acc_ta_true=c['acc_ta_true'],
                acc_st_est_mm=c['acc_st_est_mm'],
                acc_ta_est_mm=c['acc_ta_est_mm'],
                err_st=c['err_st_mm'],
                err_ta=c['err_ta_mm'],
            ))

summary_df = pd.DataFrame(rows)
display(summary_df.round(4))

# MAE per strategy averaged over datasets × test sets
mae_pivot = summary_df.groupby('strategy')[['err_st', 'err_ta']].mean().round(4)
print('\nMean MAE per strategy:')
display(mae_pivot)

In [ ]:
def plot_mae_bar(summary_df):
    strat_order = [STRATEGY_LABELS[s] for s, _ in STRATEGIES]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    for ax, metric, title in [
        (axes[0], 'err_st', 'MAE (ACC_ST, selective threshold)'),
        (axes[1], 'err_ta', 'MAE (ACC_TA, threshold-agnostic)'),
    ]:
        means = [summary_df[summary_df['strategy'] == s][metric].mean() for s in strat_order]
        stds  = [summary_df[summary_df['strategy'] == s][metric].std()  for s in strat_order]
        colors = [STRATEGY_COLORS[s] for s, _ in STRATEGIES]

        bars = ax.bar(strat_order, means, yerr=stds,
                      color=colors, capsize=5, alpha=0.85,
                      edgecolor='black', linewidth=0.8)
        for bar, mean_val in zip(bars, means):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                    f'{mean_val:.3f}', ha='center', va='bottom', fontsize=9)

        ax.set_xticks(range(len(strat_order)))
        ax.set_xticklabels(strat_order, rotation=15, ha='right')
        ax.set_ylabel('Mean Absolute Error')
        ax.set_title(title)
        ax.grid(axis='y', ls='--', alpha=0.4)
        ax.set_ylim(bottom=0)

    fig.suptitle('MAE by Decoy Strategy (entity13 + living17)', fontsize=13)
    plt.tight_layout()
    plt.show()


plot_mae_bar(summary_df)

## Визуализация 9: Большой сводный дашборд (per dataset)

In [ ]:
def plot_dashboard(breeds_name, results_dict, testset_name):
    """Дашборд 2×3: score dist, ECDF, scatter, FDR, Acc, P-R — по стратегиям."""
    tset_results = results_dict[breeds_name][testset_name]
    strat_names  = list(tset_results.keys())
    n_strats     = len(strat_names)
    n_rows       = 3  # score_dist / fdr / acc_curve

    fig, axes = plt.subplots(n_rows, n_strats, figsize=(5 * n_strats, 3.5 * n_rows))
    if n_strats == 1:
        axes = axes.reshape(n_rows, 1)

    ref = list(tset_results.values())[0]
    r   = ref['normalized_rank']
    n   = ref['n']
    cdf = np.arange(1, n + 1) / n
    all_scores = ref['pred_scores']
    bins = np.linspace(all_scores.min() - 0.3, all_scores.max() + 0.3, 60)

    for col, strat_name in enumerate(strat_names):
        c     = tset_results[strat_name]
        color = STRATEGY_COLORS[strat_name]
        inc   = c['correct'] == 0

        # Row 0: Score distribution
        ax = axes[0][col]
        sns.histplot(c['pred_scores'],  bins=bins, stat='density', color='steelblue',
                     kde=True, fill=True, alpha=0.3, label='model', ax=ax)
        sns.histplot(c['decoy_scores'], bins=bins, stat='density', color=color,
                     kde=True, fill=True, alpha=0.4, label='decoy', ax=ax)
        if inc.any():
            sns.histplot(c['pred_scores'][inc], bins=bins, stat='density',
                         color='crimson', kde=True, fill=True, alpha=0.3,
                         label='incorrect', ax=ax)
        ax.set_title(f'{STRATEGY_LABELS[strat_name]}\nacc={c["true_acc"]:.3f}', fontsize=11)
        ax.set_xlabel('Max logit'); ax.legend(fontsize=7)
        if col == 0: ax.set_ylabel('Density')

        # Row 1: FDR curves
        ax = axes[1][col]
        ax.plot(r, c['QVAL_true'],   color='gray',    lw=1.4, ls='--', label='True FDR')
        ax.plot(r, c['QVAL_mixmax'], color=color,     lw=1.8,          label='MixMax')
        ax.plot(r, c['QVAL_TDC'],    color='navy',    lw=1.2, ls=':',  label='TDC')
        ax.axhline(0.1, color='black', lw=0.7, ls=':', alpha=0.5)
        ax.set_xlabel('Frac. accepted')
        ax.set_title(f'FDR  MAE_ST={c["err_st_mm"]:.3f}', fontsize=10)
        ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3); ax.set_ylim(0, 1.05)
        if col == 0: ax.set_ylabel('q-value')

        # Row 2: Accuracy curves
        ax = axes[2][col]
        ax.plot(r, c['Acc_true'],   color='black', lw=2, ls='--', label='True Acc')
        ax.plot(r, c['Acc_est_MM'], color=color,   lw=1.8,        label='ENGPE-TA')
        ax.plot(r, c['Acc_est'],    color='navy',  lw=1.2, ls=':', label='ENGPE (TDC)')
        ax.set_xlabel('Frac. accepted')
        ax.set_title(f'Accuracy  MAE_TA={c["err_ta_mm"]:.3f}', fontsize=10)
        ax.legend(fontsize=7); ax.grid(ls='--', alpha=0.3)
        if col == 0: ax.set_ylabel('Accuracy')

    fig.suptitle(f'{breeds_name}  |  {testset_name}', fontsize=14)
    plt.tight_layout()
    plt.show()


for breeds_name in DATASETS:
    for tset_name in all_data[breeds_name]['test_sets_raw']:
        plot_dashboard(breeds_name, results, tset_name)

## Итоги

В таблице ниже — сводное MAE по всем датасетам, тестовым наборам и стратегиям.

In [ ]:
pivot = summary_df.pivot_table(
    index=['dataset', 'testset'],
    columns='strategy',
    values=['err_st', 'err_ta'],
    aggfunc='first'
).round(4)

print('MAE (ST и TA) по стратегиям:')
display(pivot)

print('\nLegend:')
print('  err_st  = |acc_est_ST - acc_true_ST|  (selective threshold — accept all)')
print('  err_ta  = |acc_est_TA - acc_true_TA|  (best threshold accuracy)')